# Tahap 1: Ingestion, Integrasi, dan Bahaya Gempa Berbasis Jarak
**Project Akhir Data Wrangling: Indeks Risiko Bencana Terpadu Kab/Kota di Jawa**

Isi tahap ini: baca semua sumber, samakan kunci wilayah ke `CC_2` (kode kab/kota 4 digit), hitung bahaya gempa berbasis jarak, lalu gabung jadi satu tabel master (100 kab/kota Jabar, Jateng, Jatim).

**Susunan folder yang dibutuhkan** (letakkan file mentah apa adanya, jangan diedit):
```
data/raw/batas_wilayah/kabkota_jabar_jateng_jatim.geojson
data/raw/usgs/raw_usgs_gempa_jawa.csv
data/raw/bps/_Jumlah_Penduduk_menurut_Kabupaten_Kota_dan_Kelompok_Umur__2026.xlsx
data/raw/gfw/IDN_salinan_GitHub.xlsx
data/raw/karhutla/luas-karhutla-kota.xlsx   (+ __1_ dan __2_)
data/raw/faskes/puskesmas.csv  dan  rumah_sakit.csv
data/raw/irbi/irbi_2024_kabkota_jawa.csv
```

In [7]:
!pip install rapidfuzz

import re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from rapidfuzz import process, fuzz

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

In [8]:
import hashlib
ROOT = Path(".")
PROV3 = ["JAWA BARAT", "JAWA TENGAH", "JAWA TIMUR"]
RADII_KM = [50, 100, 150]
RADIUS_UTAMA = 100

def cari(pola, banyak=False):
    """Cari file di semua folder kerja, tanpa peduli struktur folder atau akhiran ' (1)'. File berisi sama dibuang."""
    unik = {}
    for p in sorted(ROOT.rglob(pola)):
        if ".ipynb_checkpoints" in p.parts or "sample_data" in p.parts:
            continue
        unik.setdefault(hashlib.md5(p.read_bytes()).hexdigest(), p)
    hasil = list(unik.values())
    if not hasil:
        raise FileNotFoundError(f"Tidak ada file yang cocok dengan pola: {pola}")
    if not banyak and len(hasil) > 1:
        print(f"PERINGATAN: {pola} cocok {len(hasil)} file berbeda, dipakai: {hasil[0]}")
    print(f"{pola:45s} -> {[str(h) for h in hasil] if banyak else hasil[0]}")
    return hasil if banyak else hasil[0]

F = {
    "batas": cari("kabkota_jabar_jateng_jatim*.geojson"),
    "usgs": cari("raw_usgs_gempa_jawa*.csv"),
    "penduduk": cari("*enduduk*.xlsx"),
    "gfw": cari("IDN_salinan_GitHub*.xlsx"),
    "karhutla": cari("luas-karhutla-kota*.xlsx", banyak=True),
    "pus": cari("puskesmas*.csv"),
    "rs": cari("rumah_sakit*.csv"),
    "irbi": cari("irbi_2024_kabkota_jawa*.csv"),
}

kabkota_jabar_jateng_jatim*.geojson           -> kabkota_jabar_jateng_jatim.geojson
raw_usgs_gempa_jawa*.csv                      -> raw_usgs_gempa_jawa.csv
*enduduk*.xlsx                                -> _Jumlah Penduduk menurut Kabupaten_Kota dan Kelompok Umur, 2026.xlsx
IDN_salinan_GitHub*.xlsx                      -> IDN_salinan_GitHub.xlsx
luas-karhutla-kota*.xlsx                      -> ['luas-karhutla-kota (1).xlsx', 'luas-karhutla-kota (2).xlsx', 'luas-karhutla-kota.xlsx']
puskesmas*.csv                                -> puskesmas.csv
rumah_sakit*.csv                              -> rumah_sakit.csv
irbi_2024_kabkota_jawa*.csv                   -> irbi_2024_kabkota_jawa.csv


## 1. Fungsi bantu
Dua masalah data nyata ditangani di sini: **angka format Indonesia** (`1.284,70`) dan **nama kab/kota yang kembar** (Kabupaten Bogor vs Kota Bogor). Kunci join dibuat dari *jenis* (Kab/Kota) + nama bersih.

In [9]:
def angka_id(x):
    """'1.284,70' -> 1284.7 | '2.143,00' -> 2143.0 | '0' -> 0.0 | kosong -> NaN"""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")
    try:
        return float(s)
    except ValueError:
        return np.nan

def bersih(nama):
    s = unicodedata.normalize("NFKD", str(nama)).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"\b(KABUPATEN|KAB\.?|KOTA|ADM\.?)\b", " ", s)
    return re.sub(r"[^A-Z]", "", s)

def adalah_kota(nama):
    return str(nama).strip().upper().startswith("KOTA ")

## 2. Batas wilayah: kerangka utama (100 kab/kota)

In [10]:
kab = gpd.read_file(F["batas"])
kab["CC_2"] = kab["CC_2"].astype(str)
kab["key"] = np.where(kab["TYPE_2"].eq("Kota"), "K_", "B_") + kab["NAME_2"].map(bersih)
assert len(kab) == 100 and kab["key"].is_unique and kab["CC_2"].is_unique
KEYSET = set(kab["key"])
print(kab.crs, "|", kab["PROVINSI"].value_counts().to_dict())

def kunci_sumber(nama):
    """Kunci untuk sumber yang penulisan Kab/Kota-nya tidak konsisten (contoh GFW: 'Bandung' vs 'Kota Bandung', tapi 'Surabaya' tanpa 'Kota')."""
    b = bersih(nama)
    if adalah_kota(nama):
        return "K_" + b
    if "B_" + b in KEYSET:
        return "B_" + b
    if "K_" + b in KEYSET:
        return "K_" + b
    return "B_" + b

def petakan(df, kolom, sumber):
    """Tambah kolom CC_2. Yang tidak cocok dicoba fuzzy matching (jenis sama, skor >= 88) dan dicetak untuk dicek manual."""
    df = df.copy()
    df["key"] = df[kolom].map(kunci_sumber)
    tak = sorted(set(df.loc[~df["key"].isin(KEYSET), "key"]))
    peta = {}
    for k in tak:
        kandidat = [x for x in KEYSET if x[0] == k[0]]
        hit = process.extractOne(k, kandidat, scorer=fuzz.ratio, score_cutoff=88)
        if hit:
            peta[k] = hit[0]
    if peta:
        print(f"[{sumber}] fuzzy match (CEK MANUAL):", peta)
    df["key"] = df["key"].replace(peta)
    df = df.merge(kab[["key", "CC_2"]], on="key", how="left")
    sisa = df.loc[df["CC_2"].isna(), kolom].tolist()
    print(f"[{sumber}] {df['CC_2'].notna().sum()} baris cocok | tidak cocok: {sisa}")
    df = df[df["CC_2"].notna()]
    assert df["CC_2"].is_unique, f"{sumber}: ada CC_2 ganda"
    return df.drop(columns="key")

EPSG:4326 | {'Jawa Timur': 38, 'Jawa Tengah': 35, 'Jawa Barat': 27}


## 3. USGS: pembersihan dan bahaya gempa berbasis jarak
Masalah yang ditemukan saat eksplorasi: hanya sebagian kecil gempa jatuh di **dalam** polygon 3 provinsi karena sumber gempa Jawa banyak di laut. Jadi bahaya dihitung dari **jarak** gempa ke titik pusat tiap kab/kota, dan hasilnya dibandingkan dengan cara "titik di dalam polygon".

In [11]:
g_raw = pd.read_csv(F["usgs"])
g = g_raw[g_raw["type"].eq("earthquake")].drop_duplicates("id").copy()
g["waktu_utc"] = pd.to_datetime(g["time"], utc=True)
g["waktu_wib"] = g["waktu_utc"].dt.tz_convert("Asia/Jakarta")
g["tanggal"] = g["waktu_wib"].dt.strftime("%Y-%m-%d")
print(f"USGS mentah {len(g_raw)} baris -> setelah filter tipe & duplikat {len(g)} baris")
print("periode:", g["tanggal"].min(), "s/d", g["tanggal"].max(), "| M", g["mag"].min(), "-", g["mag"].max())

USGS mentah 933 baris -> setelah filter tipe & duplikat 933 baris
periode: 2016-01-06 s/d 2026-11-10 | M 4.0 - 7.0


In [12]:
pusat = kab.to_crs(32749).representative_point().to_crs(4326)
kab["lon_c"], kab["lat_c"] = pusat.x.values, pusat.y.values

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))

D = haversine_km(kab["lat_c"].values[:, None], kab["lon_c"].values[:, None],
                 g["latitude"].values[None, :], g["longitude"].values[None, :])   # 100 x n_gempa (km)
mag, dep = g["mag"].values[None, :], g["depth"].values[None, :]

haz = pd.DataFrame({"CC_2": kab["CC_2"].values})
for R in RADII_KM:
    haz[f"n_gempa_{R}km"] = (D <= R).sum(axis=1)
dalam = D <= RADIUS_UTAMA
haz[f"mag_max_{RADIUS_UTAMA}km"] = np.where(dalam.any(axis=1), np.where(dalam, mag, 0).max(axis=1), 0.0)
haz[f"kedalaman_rata_{RADIUS_UTAMA}km"] = np.where(dalam.sum(axis=1) > 0,
                                       (np.where(dalam, dep, 0).sum(axis=1) / np.maximum(dalam.sum(axis=1), 1)), np.nan)
haz[f"energi_rel_{RADIUS_UTAMA}km"] = (np.where(dalam, 10 ** (1.5 * (mag - 4.0)), 0)).sum(axis=1)

gg = gpd.GeoDataFrame(g, geometry=gpd.points_from_xy(g["longitude"], g["latitude"]), crs=4326)
dlm = gpd.sjoin(gg, kab[["CC_2", "geometry"]], how="inner", predicate="within")
haz["n_gempa_dalam_polygon"] = haz["CC_2"].map(dlm.groupby("CC_2").size()).fillna(0).astype(int)

print(f"Gempa yang jatuh di dalam polygon: {len(dlm)} dari {len(g)}")
print(f"Kab/kota dengan 0 gempa -> cara polygon: {(haz['n_gempa_dalam_polygon']==0).sum()} | radius {RADIUS_UTAMA} km: {(haz[f'n_gempa_{RADIUS_UTAMA}km']==0).sum()}")
haz.describe().round(2)

Gempa yang jatuh di dalam polygon: 168 dari 933
Kab/kota dengan 0 gempa -> cara polygon: 51 | radius 100 km: 0


,n_gempa_50km,n_gempa_100km,n_gempa_150km,mag_max_100km,kedalaman_rata_100km,energi_rel_100km,n_gempa_dalam_polygon
count,100.00,100.00,100.00,100.00,100.00,100.00,100.00
mean,9.22,44.67,115.10,5.60,127.42,2207.08,1.68
std,9.43,39.26,79.07,0.65,52.27,4588.83,4.39
min,0.00,8.00,34.00,4.40,33.77,21.25,0.00
25%,3.75,17.00,59.00,5.05,101.63,127.61,0.00
50%,6.00,29.00,86.00,5.75,121.46,730.39,0.00
75%,11.25,59.50,138.75,6.00,138.17,2179.40,2.00
max,58.00,181.00,342.00,7.00,369.74,41229.30,39.00


## 4. Karhutla (luas kebakaran hutan dan lahan, per kab/kota)

In [13]:
kh = pd.concat([pd.read_excel(p, dtype=str) for p in F["karhutla"]], ignore_index=True)
kh = kh[kh["Kode Wilayah"].notna()].copy()
kh["CC_2"] = kh["Kode Wilayah"].str.replace(r"\.0$", "", regex=True).str.zfill(4)
tahun = [c for c in kh.columns if re.fullmatch(r"20\d\d", str(c))]
for c in tahun:
    kh[c] = kh[c].map(angka_id)
kh = kh.rename(columns={c: f"karhutla_{c}" for c in tahun})
kolom_total = [f"karhutla_{y}" for y in range(2018, 2026)]
kh["karhutla_total_2018_2025"] = kh[kolom_total].sum(axis=1)
kh = kh[["CC_2"] + [f"karhutla_{y}" for y in tahun] + ["karhutla_total_2018_2025"]]
print(kh.shape, "kode cocok ke batas:", kh["CC_2"].isin(kab["CC_2"]).sum(), "/ 100 | duplikat:", kh["CC_2"].duplicated().sum())

(100, 11) kode cocok ke batas: 100 / 100 | duplikat: 0


## 5. Penduduk BPS 2026 (paparan)

In [14]:
raw_p = pd.read_excel(F["penduduk"], header=None)
umur = raw_p.iloc[2, 1:].astype(str).tolist()
baris, prov = [], None
for _, r in raw_p.iloc[4:].iterrows():
    nama = r[0]
    if pd.isna(nama):
        continue
    nama = str(nama).strip()
    if nama.isupper():
        prov = nama
        continue
    baris.append([prov, nama] + r[1:].tolist())
pen = pd.DataFrame(baris, columns=["provinsi", "wilayah"] + [f"u_{u}" for u in umur])
pen = pen[pen["provinsi"].isin(PROV3)].copy()
for c in pen.columns[2:]:
    pen[c] = pd.to_numeric(pen[c], errors="coerce")
pen = petakan(pen, "wilayah", "penduduk")
pen["pop_total"] = pen["u_TOTAL"]
pen["prop_anak_0_14"] = pen[["u_0-4", "u_5-9", "u_10-14"]].sum(axis=1) / pen["pop_total"]
pen["prop_lansia_65"] = pen[["u_65-69", "u_70-74", "u_75+"]].sum(axis=1) / pen["pop_total"]
pen = pen[["CC_2", "pop_total", "prop_anak_0_14", "prop_lansia_65"]]
print(pen.describe().round(3))

[penduduk] 100 baris cocok | tidak cocok: []
         pop_total  prop_anak_0_14  prop_lansia_65
count      100.000         100.000         100.000
mean   1320808.800           0.213           0.097
std     890227.799           0.016           0.020
min     122903.000           0.176           0.049
25%     806147.750           0.203           0.081
50%    1135080.000           0.213           0.097
75%    1693779.750           0.225           0.110
max    5760944.000           0.255           0.153


## 6. GFW: kehilangan tutupan pohon (tekanan lingkungan), ambang 30%

In [15]:
gf = pd.read_excel(F["gfw"], sheet_name="Subnational 2 tree cover loss")
gf = gf[(gf["threshold"].astype(str) == "30") & gf["subnational1"].str.upper().isin(PROV3)]
gf = gf[~gf["subnational2"].str.contains("Waduk", case=False)]
gf = petakan(gf, "subnational2", "GFW")
kol_loss = [f"tc_loss_ha_{y}" for y in range(2018, 2026)]
gf["gfw_loss_2018_2025_ha"] = gf[kol_loss].sum(axis=1)
gf["gfw_extent_2000_ha"] = gf["extent_2000_ha"]
gf["gfw_loss_rasio"] = np.where(gf["gfw_extent_2000_ha"] > 0, gf["gfw_loss_2018_2025_ha"] / gf["gfw_extent_2000_ha"], np.nan)
gf = gf[["CC_2", "gfw_extent_2000_ha", "gfw_loss_2018_2025_ha", "gfw_loss_rasio"]]
print("kab/kota di batas yang TIDAK ada di GFW:", kab.loc[~kab["CC_2"].isin(gf["CC_2"]), "NAME_2"].tolist())

[GFW] 99 baris cocok | tidak cocok: []
kab/kota di batas yang TIDAK ada di GFW: ['Pangandaran']


## 7. Faskes (kapasitas) dan IRBI (validasi)

In [16]:
pus = pd.read_csv(F["pus"], dtype=str)
rs = pd.read_csv(F["rs"], dtype=str)
fas = pd.DataFrame({"n_puskesmas": pus.groupby("kode_kab").size(),
                    "n_rs": rs.groupby("kode_kab").size()}).fillna(0).astype(int)
fas = fas.reindex(kab["CC_2"]).fillna(0).astype(int).rename_axis("CC_2").reset_index()

irbi = pd.read_csv(F["irbi"])
irbi = petakan(irbi, "kab_kota", "IRBI")[["CC_2", "skor", "kelas"]].rename(columns={"skor": "irbi_skor", "kelas": "irbi_kelas"})
print(fas.describe().round(1))

[IRBI] 100 baris cocok | tidak cocok: []
       n_puskesmas   n_rs
count        100.0  100.0
mean          29.4   11.7
std           15.7   10.6
min            3.0    1.0
25%           21.0    6.0
50%           27.0    8.0
75%           36.2   12.2
max          101.0   61.0


## 8. Gabung jadi 1 tabel queke care gap dan laporan kualitas data

In [17]:
master = pd.DataFrame(kab.drop(columns=["geometry", "key"])[["CC_2", "NAME_1", "NAME_2", "TYPE_2", "lon_c", "lat_c"]])
for t in [haz, kh, pen, gf, fas, irbi]:
    master = master.merge(t, on="CC_2", how="left", validate="one_to_one")
master["faskes_total"] = master["n_puskesmas"] + master["n_rs"]
master["faskes_per_100k"] = master["faskes_total"] / master["pop_total"] * 100_000

assert len(master) == 100
print(master.shape)
laporan = (master.isna().sum().rename("missing").to_frame()
           .assign(persen=lambda d: (d["missing"] / len(master) * 100).round(1))
           .query("missing > 0"))
print("Kolom dengan missing value:"); print(laporan)
master.head()

(100, 35)
Kolom dengan missing value:
                       missing  persen
gfw_extent_2000_ha           1     1.0
gfw_loss_2018_2025_ha        1     1.0
gfw_loss_rasio               1     1.0


,CC_2,NAME_1,NAME_2,TYPE_2,lon_c,lat_c,n_gempa_50km,n_gempa_100km,n_gempa_150km,mag_max_100km,kedalaman_rata_100km,energi_rel_100km,n_gempa_dalam_polygon,karhutla_2018,karhutla_2019,karhutla_2020,karhutla_2021,karhutla_2022,karhutla_2023,karhutla_2024,karhutla_2025,karhutla_2026,karhutla_total_2018_2025,pop_total,prop_anak_0_14,prop_lansia_65,gfw_extent_2000_ha,gfw_loss_2018_2025_ha,gfw_loss_rasio,n_puskesmas,n_rs,irbi_skor,irbi_kelas,faskes_total,faskes_per_100k
0,3204,Jawa Barat,Bandung,Kabupaten,107.632474,-7.063244,21,120,296,6.5,77.302592,7812.729911,11,0.0,494.0,0.0,0.0,0.0,132.16,12.45,0.0,25.16,638.61,3812412,0.235873,0.067161,106947.0,1924.0,0.017990,62,11,117.13,SEDANG,73,1.914798
1,3217,Jawa Barat,Bandung Barat,Kabupaten,107.412727,-6.907284,27,99,294,5.8,78.390859,1896.987644,1,0.0,0.0,33.0,0.0,23.0,0.00,33.54,0.0,15.97,89.54,1931072,0.234400,0.073748,75471.0,698.0,0.009249,32,9,108.28,SEDANG,41,2.123173
2,3279,Jawa Barat,Banjar,Kota,108.560833,-7.380248,14,95,209,6.5,97.706916,6464.916430,0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,214084,0.221754,0.100264,7315.0,98.0,0.013397,10,4,116.26,SEDANG,14,6.539489
3,3216,Jawa Barat,Bekasi,Kabupaten,107.158836,-6.198461,11,59,155,5.8,155.765322,1202.993354,2,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,5.85,0.00,3331400,0.235062,0.049241,9187.0,96.0,0.010450,44,53,106.39,SEDANG,97,2.911689
4,3201,Jawa Barat,Bogor,Kabupaten,107.001897,-6.545627,22,104,234,5.8,105.535683,1516.784811,6,0.0,0.0,0.0,0.0,0.0,0.00,37.73,0.0,20.91,37.73,5760944,0.235982,0.057962,180857.0,4953.0,0.027386,101,30,101.72,SEDANG,131,2.273933


In [19]:
Path("data/interim").mkdir(parents=True, exist_ok=True)
master.to_csv("data/interim/queke care gap.csv", index=False)
print("tersimpan: data/interim/queke care gap.csv")

tersimpan: data/interim/queke care gap.csv


## Catatan untuk laporan (tulis alasan tiap keputusan)
1. **Bahaya gempa berbasis jarak**, bukan titik-di-polygon, karena sebagian besar gempa terjadi di laut. Bandingkan hasil kedua cara dan uji beberapa radius (50, 100, 150 km) di tahap sensitivitas.
2. **Kunci join `CC_2`** dipakai untuk batas, karhutla, dan faskes. Penduduk, GFW, dan IRBI hanya punya nama, jadi dicocokkan dengan aturan Kab/Kota plus fuzzy matching yang dicek manual.
3. **Karhutla 2026 tidak dimasukkan ke total** karena belum setahun penuh.
4. **Missing value** (contoh kab yang tidak ada di GFW) belum ditangani di tahap ini. Keputusan imputasi atau hapus ditulis di tahap profiling dan cleansing.